##Imports

In [11]:
#Data Manipulation and Visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Data Preprocessing

from sklearn.preprocessing import OneHotEncoder,LabelEncoder, OrdinalEncoder,TargetEncoder
from sklearn.preprocessing import StandardScaler,MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

#Pipeline

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

#Model Building

from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor

#Evaluation Metrics

from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score

In [12]:
import sklearn
print(sklearn.__version__)

1.9.1


In [13]:
#Importing the cleaned data

df = pd.read_csv('../data/youtube_df_cleaned.csv')

In [14]:
#Initial health check of the dataset

df.shape

#(120000,13)

(120000, 13)

In [15]:
df.head()

,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,category,device,country,ad_revenue_usd,comment_rate,engagement_rate,watch_time_per_view
0,9936,1221,320,26497.214184,2.862137,228086,Entertainment,TV,IN,203.178237,0.032206,0.155093,2.666789
1,10017,642,346,15209.747445,23.738069,736015,Gaming,Tablet,CA,140.880508,0.034541,0.098632,1.518393
2,10097,1979,187,57332.658498,26.200634,240534,Education,TV,CA,360.134008,0.018520,0.214519,5.678187
3,10034,1191,242,31334.517771,11.770340,434482,Entertainment,Mobile,UK,224.638261,0.024118,0.142814,3.122834
4,9889,1858,477,15665.666434,6.635854,42030,Education,Mobile,CA,165.514388,0.048235,0.236121,1.584151


In [16]:
df.isna().sum()

views                   0
likes                   0
comments                0
watch_time_minutes      0
video_length_minutes    0
subscribers             0
category                0
device                  0
country                 0
ad_revenue_usd          0
comment_rate            0
engagement_rate         0
watch_time_per_view     0
dtype: int64

In [17]:
df.duplicated().sum()

np.int64(0)

In [18]:
#Derived features introduced multicollinearity due to mathematical dependency,
#so I retained base features to maintain model stability

df_model = df.drop(['comment_rate','watch_time_per_view'],axis=1)
df_model.shape

(120000, 11)

In [19]:
df_model.head()

,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,category,device,country,ad_revenue_usd,engagement_rate
0,9936,1221,320,26497.214184,2.862137,228086,Entertainment,TV,IN,203.178237,0.155093
1,10017,642,346,15209.747445,23.738069,736015,Gaming,Tablet,CA,140.880508,0.098632
2,10097,1979,187,57332.658498,26.200634,240534,Education,TV,CA,360.134008,0.214519
3,10034,1191,242,31334.517771,11.770340,434482,Entertainment,Mobile,UK,224.638261,0.142814
4,9889,1858,477,15665.666434,6.635854,42030,Education,Mobile,CA,165.514388,0.236121


##Splitting Feature Columns(X) and Target Column(Y)

In [20]:
df_model.columns

Index(['views', 'likes', 'comments', 'watch_time_minutes',
       'video_length_minutes', 'subscribers', 'category', 'device', 'country',
       'ad_revenue_usd', 'engagement_rate'],
      dtype='str')

In [21]:
#Splitting X and Y

X=df_model.drop(['ad_revenue_usd'],axis=1)
Y=df_model['ad_revenue_usd']

In [22]:
X

,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,category,device,country,engagement_rate
0,9936,1221,320,26497.214184,2.862137,228086,Entertainment,TV,IN,0.155093
1,10017,642,346,15209.747445,23.738069,736015,Gaming,Tablet,CA,0.098632
2,10097,1979,187,57332.658498,26.200634,240534,Education,TV,CA,0.214519
3,10034,1191,242,31334.517771,11.770340,434482,Entertainment,Mobile,UK,0.142814
4,9889,1858,477,15665.666434,6.635854,42030,Education,Mobile,CA,0.236121
...,...,...,...,...,...,...,...,...,...,...
119995,9853,1673,147,42075.704885,25.490195,210818,Education,Tablet,US,0.184715
119996,10128,1709,63,57563.703040,16.229133,878860,Music,Desktop,UK,0.174961
119997,10267,700,0,27549.714659,23.822365,576756,Tech,Tablet,CA,0.068182
119998,10240,1616,106,56967.384382,7.753099,585138,Music,Mobile,UK,0.168164


In [23]:
Y

0         203.178237
1         140.880508
2         360.134008
3         224.638261
4         165.514388
             ...    
119995    280.986396
119996    354.612981
119997    203.643106
119998    351.525811
119999    253.842824
Name: ad_revenue_usd, Length: 120000, dtype: float64

##Splitting Train and Test Data



In [24]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)
X_train.shape,X_test.shape,Y_train.shape,Y_test.shape

((96000, 10), (24000, 10), (96000,), (24000,))

In [25]:
df_model.columns

Index(['views', 'likes', 'comments', 'watch_time_minutes',
       'video_length_minutes', 'subscribers', 'category', 'device', 'country',
       'ad_revenue_usd', 'engagement_rate'],
      dtype='str')

In [26]:
Num_Col = X.select_dtypes(include=['int64','float64'])
Num_Col

,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,engagement_rate
0,9936,1221,320,26497.214184,2.862137,228086,0.155093
1,10017,642,346,15209.747445,23.738069,736015,0.098632
2,10097,1979,187,57332.658498,26.200634,240534,0.214519
3,10034,1191,242,31334.517771,11.770340,434482,0.142814
4,9889,1858,477,15665.666434,6.635854,42030,0.236121
...,...,...,...,...,...,...,...
119995,9853,1673,147,42075.704885,25.490195,210818,0.184715
119996,10128,1709,63,57563.703040,16.229133,878860,0.174961
119997,10267,700,0,27549.714659,23.822365,576756,0.068182
119998,10240,1616,106,56967.384382,7.753099,585138,0.168164


In [27]:
Cat_Col = X.select_dtypes(include='object')
Cat_Col

C:\Users\sange\AppData\Local\Temp\ipykernel_41316\4008300311.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  Cat_Col = X.select_dtypes(include='object')


,category,device,country
0,Entertainment,TV,IN
1,Gaming,Tablet,CA
2,Education,TV,CA
3,Entertainment,Mobile,UK
4,Education,Mobile,CA
...,...,...,...
119995,Education,Tablet,US
119996,Music,Desktop,UK
119997,Tech,Tablet,CA
119998,Music,Mobile,UK


In [28]:
#Numerical Transformer

Num_trans = Pipeline(steps=[
    ('Imputing',SimpleImputer(strategy='median')),
    ('Scaling',MinMaxScaler()),
])

#Categorical tranformer

Cat_trans = Pipeline(steps=[
    ('Imputing',SimpleImputer(strategy='most_frequent')),
    ('Encoding',OneHotEncoder())
])

In [29]:
#Column Transformer

Pre_Process_Step = ColumnTransformer(transformers=[
    ('Numerical_Transformer',Num_trans,Num_Col.columns),
    ('Categorical_Transformer',Cat_trans,Cat_Col.columns),
])

Pre_Process_Step

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numerical_Transformer', ...), ('Categorical_Transformer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``t

## 1) Linear Regression Model

In [30]:
#Pipeline

LR_Pipeline = Pipeline(steps=[
    ('Pre_Processing',Pre_Process_Step),
    ('Linear_Regression',LinearRegression())
])

LR_Pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Pre_Processing', ...), ('Linear_Regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numerical_Transformer', ...), ('Categorical_Transformer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_

In [31]:
LR_Pipeline.fit(X_train,Y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Pre_Processing', ...), ('Linear_Regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['views','likes','comments',...,'device','country','engagement_rate']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numerical_Transformer', ...), ('Categorical_Transformer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be au

In [32]:
#Prediction

Y_train_pred = LR_Pipeline.predict(X_train)
Y_test_pred = LR_Pipeline.predict(X_test)



In [33]:
#Model Evaluation

#MAE for Train and Test data

MAE_train = mean_absolute_error(Y_train,Y_train_pred)
MAE_test=mean_absolute_error(Y_test,Y_test_pred)
print('Train MAE',MAE_train)
print('Test MAE',MAE_test)
print('\n')

#MSE fro train and test data

MSE_train = mean_squared_error(Y_train,Y_train_pred)
MSE_test = mean_squared_error(Y_test,Y_test_pred)
#print('Train MSE',MSE_train)
#print('Test MSE',MSE_test)
#print('\n')

#RMSE for train and test data

RMSE_train = np.sqrt(MSE_train)
RMSE_test = np.sqrt(MSE_test)
print('Train RMSE',RMSE_train)
print('Test RMSE',RMSE_test)



Train MAE 3.513124013721291
Test MAE 3.365221054821165


Train RMSE 13.926958864326705
Test RMSE 13.537684852184087


In [34]:
#Performance Metric

#R2 Score

R2_train = r2_score(Y_train,Y_train_pred)
R2_test = r2_score(Y_test,Y_test_pred)
print('Train R2 Score',R2_train)
print('Test R2 Score',R2_test)


Train R2 Score 0.9494893111545485
Test R2 Score 0.95216753344343


My LinearRegression model achieved around 95% R² on both train and test data, indicating strong performance with no overfitting, and low MAE shows accurate predictions.

In [35]:
LR_Pipeline.steps[1][1].coef_

array([ 4.18432962e+00,  4.70436271e+01,  1.17815658e+01,  2.21149699e+02,
        7.49976512e-02,  7.53079061e-02, -1.91212197e+01,  2.53711607e-02,
        7.06859675e-03, -7.98840053e-02,  3.92072074e-02, -6.85680340e-02,
        7.68050745e-02,  9.92784489e-02,  2.79835792e-02, -1.33922070e-01,
        6.66004211e-03,  3.94270297e-02,  1.58233451e-02, -7.42807926e-02,
        9.18780023e-02, -2.90131208e-02, -4.38344637e-02])

In [36]:
# Feature names after preprocessing
feature_names = LR_Pipeline.named_steps['Pre_Processing'].get_feature_names_out()

# Coefficients
coefficients = LR_Pipeline.named_steps['Linear_Regression'].coef_

# Dataframe
import pandas as pd

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Absolute values
coef_df['abs_coef'] = coef_df['Coefficient'].abs()

# Sorting by importance
coef_df = coef_df.sort_values(by='abs_coef', ascending=False)

# Top features
coef_df.head(10)

,Feature,Coefficient,abs_coef
3,Numerical_Transformer__watch_time_minutes,221.149699,221.149699
1,Numerical_Transformer__likes,47.043627,47.043627
6,Numerical_Transformer__engagement_rate,-19.121220,19.121220
2,Numerical_Transformer__comments,11.781566,11.781566
0,Numerical_Transformer__views,4.184330,4.184330
15,Categorical_Transformer__device_TV,-0.133922,0.133922
13,Categorical_Transformer__device_Desktop,0.099278,0.099278
20,Categorical_Transformer__country_IN,0.091878,0.091878
9,Categorical_Transformer__category_Gaming,-0.079884,0.079884
12,Categorical_Transformer__category_Tech,0.076805,0.076805


#Insight :
Watch time is the most important feature, showing a strong positive impact on revenue.
Likes and comments also positively influence revenue, indicating user engagement matters.
Views have a smaller effect, suggesting engagement quality is more important than quantity.
Engagement rate shows a negative impact due to multicollinearity with likes and comments.
Categorical features like device, country, and category have minimal influence compared to behavioral features.

## 2) Ridge Regression Model

In [37]:
from sklearn.linear_model import Ridge

ridge_pipeline = Pipeline(steps=[
    ('Preprocessing', Pre_Process_Step),
    ('Model', Ridge(alpha=0.9))
])

# Train
ridge_pipeline.fit(X_train, Y_train)

# Predict
Y_train_pred = ridge_pipeline.predict(X_train)
Y_test_pred = ridge_pipeline.predict(X_test)

# Evaluate

print("Train R2:", r2_score(Y_train, Y_train_pred))
print("Test R2:", r2_score(Y_test, Y_test_pred))

Train R2: 0.9494883564615143
Test R2: 0.9521721414906903





*   The Ridge Regression model achieved an R² score of ~0.95 on both train and test data, indicating strong predictive performance. The similar scores suggest that the model is well-generalized and not overfitting.
*   Regularization helped stabilize the coefficients without sacrificing accuracy.



## 3) Lasso Regression



In [38]:
from sklearn.linear_model import Lasso

lasso_pipeline = Pipeline(steps=[
    ('Preprocessing', Pre_Process_Step),
    ('Model', Lasso(alpha=0.1))
])

# Train
lasso_pipeline.fit(X_train, Y_train)

# Predict
Y_train_pred = lasso_pipeline.predict(X_train)
Y_test_pred = lasso_pipeline.predict(X_test)

# Evaluate

print("Train R2:", r2_score(Y_train, Y_train_pred))
print("Test R2:", r2_score(Y_test, Y_test_pred))

Train R2: 0.9492930608284973
Test R2: 0.9520359727029106




*   Lasso Regression achieved an R² score of ~0.95 on both train and test data, indicating strong predictive performance.
*   It also helps in feature selection by shrinking less important feature coefficients, making the model more interpretable.







*   Even though Linear Regression performed well, I applied Ridge and Lasso to handle multicollinearity and compare model robustness.
*    This helps ensure stability and better feature selection



## 4) Random Forest Regressor

In [39]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('Preprocessing', Pre_Process_Step),
    ('Model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train
rf_pipeline.fit(X_train, Y_train)

# Predict
Y_train_pred = rf_pipeline.predict(X_train)
Y_test_pred = rf_pipeline.predict(X_test)

# Evaluate

print("Train R2:", r2_score(Y_train, Y_train_pred))
print("Test R2:", r2_score(Y_test, Y_test_pred))

Train R2: 0.9925428081585805
Test R2: 0.948988578180241




*   Random Forest achieved a very high R² score (~0.99 on train and ~0.95 on test), indicating strong predictive performance
*   The slight drop from train to test suggests minor overfitting, but overall the model generalizes well.



## 5) Gradient Boosting Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline(steps=[
    ('Preprocessing', Pre_Process_Step),
    ('Model', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
])

# Train
gb_pipeline.fit(X_train, Y_train)

# Predict
Y_train_pred = gb_pipeline.predict(X_train)
Y_test_pred = gb_pipeline.predict(X_test)

# Evaluate

print("Train R2:", r2_score(Y_train, Y_train_pred))
print("Test R2:", r2_score(Y_test, Y_test_pred))


*   Gradient Boosting achieved consistent R² scores (~0.95) on both train and test data, indicating strong and stable performance.
*   The nearly equal scores show that the model generalizes well with minimal overfitting.



Although Random Forest achieved a higher train R², it showed slight overfitting compared to Linear/Ridge models. Linear Regression and Ridge provided almost equal and high performance (~95% R²) on both train and test data, indicating better generalization and stability. Therefore, a simpler linear model is preferred for this problem.

Since the goal is to predict ad revenue and understand feature impact, Linear Regression is preferred due to its interpretability. It clearly shows how each feature contributes to revenue, which is useful for business decision-making. Additionally, it provides stable and accurate predictions without overfitting.

In [ ]:
import joblib
import sklearn

print("Scikit-learn version:", sklearn.__version__)

joblib.dump(LR_Pipeline, '../app/model_pipeline.pkl')

print("Model saved successfully!")

['../app/model_pipeline.pkl']